# Exchangeability Analysis Notebook
Run exchangeability analysis (if needed) and inspect the resulting CSV.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
import pandas as pd

In [ ]:
def _find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for p in [cur] + list(cur.parents):
        if (p / 'pyproject.toml').exists():
            return p
    raise FileNotFoundError('Could not locate repo root (missing pyproject.toml).')

REPO_ROOT = _find_repo_root(Path.cwd())
BASE_SAVE_DIR = Path(os.environ.get('BASE_SAVE_DIR', '/n/netscratch/kempner_pehlevan_lab/Lab/ilavie/exchangeability_outputs'))
RUN_ID = 'exchangeability'
RUN_ID_RESOLUTION = 'latest_prefix'  # latest_prefix picks newest RUN_ID_* dir; use exact for a fixed folder
WIDTHS = [32,64,128,256,512]  # set to [] to analyze all widths
SHUFFLE_REPEATS = 2000
PROBE_BATCH_SIZE = 1024
PROBE_LOADER_BATCH_SIZE = 128
LOG_EVERY_SHUFFLES = 50
WRITE_EVERY_SHUFFLES = 1
RUN_ANALYSIS = True  # set False to skip running and only inspect existing CSV
RESUME = True  # True: continue from existing CSV, False: recompute from scratch

# Keep output CSV stable across WIDTHS changes so resume can skip already-computed widths.
OUTPUT_CSV_OVERRIDE = os.environ.get('EXCHANGEABILITY_OUTPUT_CSV', '').strip()
if OUTPUT_CSV_OVERRIDE:
    override_path = Path(OUTPUT_CSV_OVERRIDE)
    OUTPUT_CSV = override_path if override_path.is_absolute() else (BASE_SAVE_DIR / override_path)
else:
    default_csv = BASE_SAVE_DIR / 'exchangeability_metrics.csv'
    if default_csv.exists():
        OUTPUT_CSV = default_csv
    else:
        candidates = sorted(
            [p for p in BASE_SAVE_DIR.glob('exchangeability_metrics*.csv') if '_summary' not in p.stem],
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        )
        OUTPUT_CSV = candidates[0] if candidates else default_csv

SIMILARITY_OUTPUT_DIR = OUTPUT_CSV.with_name(OUTPUT_CSV.stem + '_similarity')

print(f'REPO_ROOT={REPO_ROOT}')
print(f'BASE_SAVE_DIR={BASE_SAVE_DIR}')
print(f'OUTPUT_CSV={OUTPUT_CSV}')
print(f'SIMILARITY_OUTPUT_DIR={SIMILARITY_OUTPUT_DIR}')
print(f'RUN_ID_RESOLUTION={RUN_ID_RESOLUTION}')


In [ ]:
if RUN_ANALYSIS:
    cmd = [
        'analyze_exchangeability.py',
        '--base-save-dir', str(BASE_SAVE_DIR),
        '--run-id', RUN_ID,
        '--run-id-resolution', RUN_ID_RESOLUTION,
        '--output-csv', str(OUTPUT_CSV),
        '--similarity-output-dir', str(SIMILARITY_OUTPUT_DIR),
        '--shuffle-repeats', str(SHUFFLE_REPEATS),
        '--probe-batch-size', str(PROBE_BATCH_SIZE),
        '--probe-loader-batch-size', str(PROBE_LOADER_BATCH_SIZE),
        '--log-every-shuffles', str(LOG_EVERY_SHUFFLES),
        '--write-every-shuffles', str(WRITE_EVERY_SHUFFLES),
    ]
    if RESUME:
        cmd.append('--resume')
    else:
        cmd.append('--no-resume')

    if WIDTHS:
        cmd.extend(['--widths'] + [str(w) for w in WIDTHS])

    print('Running (in-process):')
    print(' '.join([str(REPO_ROOT / 'scripts' / cmd[0])] + cmd[1:]))

    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))
    from scripts.analyze_exchangeability import main as analyze_main

    prev_argv = sys.argv[:]
    prev_cwd = Path.cwd()
    try:
        os.chdir(REPO_ROOT)
        sys.argv = cmd
        analyze_main()
    finally:
        sys.argv = prev_argv
        os.chdir(prev_cwd)
else:
    print(f'Skipping analysis by request (RUN_ANALYSIS=False). Using CSV: {OUTPUT_CSV}')


In [ ]:
df = pd.read_csv(OUTPUT_CSV)
print(f'rows={len(df)}, widths={sorted(df["width"].unique().tolist())}')
df.head()

In [ ]:
coverage = (
    df.groupby(['width', 'representation', 'analysis_type'], as_index=False)
      .agg(num_points=('images_seen', 'nunique'), min_images_seen=('images_seen', 'min'), max_images_seen=('images_seen', 'max'))
      .sort_values(['width', 'representation', 'analysis_type'])
)
coverage

In [ ]:
summary = (
    df.groupby(['representation', 'analysis_type', 'width', 'images_seen'], as_index=False)
      .agg(
          ks_distance_mean=('ks_distance', 'mean'),
          w1_distance_mean=('w1_distance', 'mean'),
          train_loss=('train_loss', 'mean'),
          val_loss=('val_loss', 'mean'),
          train_error=('train_error', 'mean'),
          val_error=('val_error', 'mean'),
      )
      .sort_values(['width', 'images_seen', 'representation', 'analysis_type'])
)
summary_path = OUTPUT_CSV.with_name(OUTPUT_CSV.stem + '_summary.csv')
summary.to_csv(summary_path, index=False)
print(f'Wrote {summary_path}')
summary.head()

In [ ]:
observed = df[(df['analysis_type'] == 'within_vs_across_real') & (df['shuffle_id'] == -1)].copy()
cols = [
    'width',
    'images_seen',
    'representation',
    'ks_distance',
    'ks_p_raw',
    'ks_sigma_two_sided',
    'ks_p_empirical',
    'ks_sigma_empirical_two_sided',
    'w1_distance',
    'w1_p_empirical',
    'w1_sigma_empirical_two_sided',
]
available_cols = [c for c in cols if c in observed.columns]
observed = observed[available_cols].sort_values(['width', 'images_seen', 'representation'])
print(f'Observed rows: {len(observed)}')
observed.head(50)